In [1]:
from Booleanize_global import *

In [2]:
from Booleanize_global import *
import os
import numpy as np

csv_dir = "fractal_maps_fmnist_M8_S1"   # 🔥 changed

csv_files = [f for f in os.listdir(csv_dir) if f.endswith(".csv")]

print("Total CSV files:", len(csv_files))  # ~70000

Total CSV files: 70000


In [3]:
# =========================
# SPLIT TRAIN / TEST
# =========================
train_files = []
test_files = []

for f in csv_files:
    parts = f.split("_")

    split = parts[0]      # train / test
    label = int(parts[1]) # 0–9

    if split == "train":
        train_files.append(f)
    else:
        test_files.append(f)

print("Train:", len(train_files))
print("Test:", len(test_files))

Train: 60000
Test: 10000


In [4]:
# =========================
# COLLECT TRAIN VALUES
# =========================
all_train_values = []

for f in train_files:
    path = os.path.join(csv_dir, f)

    vals = np.loadtxt(path, delimiter=",").flatten()
    all_train_values.extend(vals)

print("Collected training fractal values:", len(all_train_values))

Collected training fractal values: 194940000


In [5]:
# =========================
# FIT GLOBAL BINS
# =========================
discretizer = fit_global_bins(all_train_values, n_bins=3)

print("\nGlobal Quantile Bin Boundaries:")
bins = discretizer.bin_edges_[0]

for i in range(len(bins)-1):
    print(f"Bin {i}: {bins[i]:.5f} → {bins[i+1]:.5f}")


Global Quantile Bin Boundaries:
Bin 0: -1.00000 → 2.00000
Bin 1: 2.00000 → 2.13386
Bin 2: 2.13386 → 5.00000


/home/ubuntu/local/fractal_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [6]:
# =========================
# BOOLEANIZATION
# =========================
booleanized_data = {}

for idx, f in enumerate(csv_files):

    path = os.path.join(csv_dir, f)
    vals = np.loadtxt(path, delimiter=",").flatten()

    booleanized_data[f] = booleanize_array(vals, discretizer).flatten()

    if (idx + 1) % 2000 == 0:
        print(f"{idx+1}/{len(csv_files)} processed")

print("Booleanization completed.")

2000/70000 processed
4000/70000 processed
6000/70000 processed
8000/70000 processed
10000/70000 processed
12000/70000 processed
14000/70000 processed
16000/70000 processed
18000/70000 processed
20000/70000 processed
22000/70000 processed
24000/70000 processed
26000/70000 processed
28000/70000 processed
30000/70000 processed
32000/70000 processed
34000/70000 processed
36000/70000 processed
38000/70000 processed
40000/70000 processed
42000/70000 processed
44000/70000 processed
46000/70000 processed
48000/70000 processed
50000/70000 processed
52000/70000 processed
54000/70000 processed
56000/70000 processed
58000/70000 processed
60000/70000 processed
62000/70000 processed
64000/70000 processed
66000/70000 processed
68000/70000 processed
70000/70000 processed
Booleanization completed.


In [7]:
# =========================
# CREATE TRAIN / TEST SETS
# =========================
X_train, X_test = [], []
Y_train, Y_test = [], []

for f in train_files:
    parts = f.split("_")
    label = int(parts[1])

    X_train.append(booleanized_data[f])
    Y_train.append(label)

for f in test_files:
    parts = f.split("_")
    label = int(parts[1])

    X_test.append(booleanized_data[f])
    Y_test.append(label)

X_train = np.array(X_train)
X_test  = np.array(X_test)
Y_train = np.array(Y_train)
Y_test  = np.array(Y_test)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Shapes:
X_train: (60000, 16245)
X_test : (10000, 16245)


In [8]:
# =========================
# SAVE FILES
# =========================
np.save('X_train_fmnist.npy', X_train)   # 🔥 renamed (recommended)
np.save('X_test_fmnist.npy', X_test)
np.save('Y_train_fmnist.npy', Y_train)
np.save('Y_test_fmnist.npy', Y_test)

print("\n✅ Saved FMNIST booleanized dataset!")


✅ Saved FMNIST booleanized dataset!


In [9]:
X_train[5]

array([0, 1, 0, ..., 0, 0, 0], shape=(16245,))

In [10]:
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(60000, 16245) (60000,)
(10000, 16245) (10000,)


In [11]:
print("Train:", np.bincount(Y_train))
print("Test:", np.bincount(Y_test))

Train: [6000 6000 6000 6000 6000 6000 6000 6000 6000 6000]
Test: [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]
